In [2]:
import pandas as pd
from binance.client import Client
client = Client()

In [3]:
info = client.get_exchange_info()
df_pairs = pd.DataFrame(info["symbols"])
df_active = df_pairs[df_pairs["status"] == "TRADING"][
    ["symbol", "baseAsset", "quoteAsset"]
]


In [4]:
all_tickers = Client().get_all_tickers()
df_tickers = pd.DataFrame(all_tickers)
df_tickers["price"] = df_tickers["price"].astype(float)

In [5]:
df_usdt = df_active[df_active['quoteAsset']=="USDT"]

In [6]:
df_merged = pd.merge(df_usdt, df_tickers, on="symbol", how="left")

In [7]:
## Known altcoins close to USDT price
altcoins = ["XRP", "DOT", "AXS", "GAS", "PROM", "PENDLE", "ORCA", "VANA", "U", "ATM", "ASR", "MOVR", "CVX", "RPL", "ZRO", "EUL"]

In [8]:
stable_coins = df_merged[(df_merged["price"] < 1.3) & (df_merged["price"] > 0.8)]['baseAsset'].to_list()
stable_coins  = [i for i in stable_coins if i not in altcoins]
stable_coins

['TUSD',
 'USDC',
 'EUR',
 'CAKE',
 'USDP',
 'FDUSD',
 'AEUR',
 'EURI',
 'XUSD',
 'USD1',
 'BFUSD',
 'USDE',
 'RLUSD',
 'USDS']

In [9]:
stable_coin_df = df_active[(df_active['quoteAsset']=="USDT") & df_active["baseAsset"].isin(stable_coins)].reset_index(drop=True)
stable_coin_symbols = stable_coin_df["symbol"].to_list()
stable_coin_symbols

['TUSDUSDT',
 'USDCUSDT',
 'EURUSDT',
 'CAKEUSDT',
 'USDPUSDT',
 'FDUSDUSDT',
 'AEURUSDT',
 'EURIUSDT',
 'XUSDUSDT',
 'USD1USDT',
 'BFUSDUSDT',
 'USDEUSDT',
 'RLUSDUSDT',
 'USDSUSDT']

In [10]:
stable_coins.append("USDT")
tri_arb_assets_df = df_active[df_active["quoteAsset"].isin(stable_coins)].reset_index(drop=True)
tri_arb_assets_df

,symbol,baseAsset,quoteAsset
0,BTCUSDT,BTC,USDT
1,ETHUSDT,ETH,USDT
2,BNBUSDT,BNB,USDT
3,NEOUSDT,NEO,USDT
4,LTCUSDT,LTC,USDT
...,...,...,...
813,AIGENSYNUSDC,AIGENSYN,USDC
814,GENIUSUSDT,GENIUS,USDT
815,GENIUSUSDC,GENIUS,USDC
816,OPGUSDT,OPG,USDT


In [11]:
tri_arb_assets_df.groupby("quoteAsset")["symbol"].count()

quoteAsset
EUR       29
EURI       3
FDUSD     37
RLUSD      1
USD1      27
USDC     289
USDS       2
USDT     430
Name: symbol, dtype: int64

In [12]:
stable_coins_low_vol = [i for i in stable_coins if i not in ["USDT", "USDC"]]
tri_arb_low_vol_assets_df = df_active[df_active["quoteAsset"].isin(stable_coins_low_vol)].reset_index(drop=True)
tri_arb_low_vol_assets_df

,symbol,baseAsset,quoteAsset
0,BTCEUR,BTC,EUR
1,ETHEUR,ETH,EUR
2,BNBEUR,BNB,EUR
3,XRPEUR,XRP,EUR
4,LINKEUR,LINK,EUR
...,...,...,...
94,ETHUSDS,ETH,USDS
95,币安人生USD1,币安人生,USD1
96,CHIPUSD1,CHIP,USD1
97,XAUTUSD1,XAUT,USD1


In [13]:
all_24h_stats = client.get_ticker()
df_all = pd.DataFrame(all_24h_stats)
all_vol = df_all[["symbol", "weightedAvgPrice", "volume"]].astype({"weightedAvgPrice": float, "volume": float})

all_vol["$vol"] = all_vol["weightedAvgPrice"]*all_vol["volume"]

In [14]:
universe_wo_vol = pd.merge(tri_arb_low_vol_assets_df, all_vol, on="symbol", how="left")
universe_wo_vol

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume,$vol
0,BTCEUR,BTC,EUR,53419.936211,3.521822e+02,1.881355e+07
1,ETHEUR,ETH,EUR,1416.709961,6.070593e+03,8.600270e+06
2,BNBEUR,BNB,EUR,510.083155,1.040735e+03,5.308614e+05
3,XRPEUR,XRP,EUR,0.968426,3.124875e+06,3.026211e+06
4,LINKEUR,LINK,EUR,6.699281,2.054164e+04,1.376142e+05
...,...,...,...,...,...,...
94,ETHUSDS,ETH,USDS,1637.758503,6.282510e+01,1.028923e+05
95,币安人生USD1,币安人生,USD1,0.709817,1.591820e+04,1.129902e+04
96,CHIPUSD1,CHIP,USD1,0.033464,2.185940e+05,7.315014e+03
97,XAUTUSD1,XAUT,USD1,4188.869292,5.566800e+00,2.331860e+04


In [15]:
ex_list = df_active[(df_active["quoteAsset"]=="USDT") & (df_active["baseAsset"].isin(stable_coins_low_vol))]["symbol"].to_list()
stable_ex_df = all_vol[all_vol["symbol"].isin(ex_list)]
stable_ex_df.rename(columns={"symbol":"ex_symbol", "weightedAvgPrice": "usdtPrice", "volume": "ex_volume"}, inplace=True)
stable_ex_df["quoteAsset"] = stable_ex_df["ex_symbol"].str[:-4]
stable_ex_df.reset_index(drop=True, inplace=True)

stable_ex_df

,ex_symbol,usdtPrice,ex_volume,$vol,quoteAsset
0,TUSDUSDT,1.000046,1.350300e+05,1.350362e+05,TUSD
1,EURUSDT,1.155644,1.952146e+07,2.255985e+07,EUR
2,CAKEUSDT,1.306800,1.423449e+06,1.860163e+06,CAKE
3,USDPUSDT,1.000181,4.444000e+03,4.444805e+03,USDP
4,FDUSDUSDT,0.998217,2.008772e+07,2.005191e+07,FDUSD
5,AEURUSDT,1.148436,7.192000e+02,8.259551e+02,AEUR
6,EURIUSDT,1.155142,1.730533e+06,1.999012e+06,EURI
7,XUSDUSDT,1.000829,2.877336e+06,2.879722e+06,XUSD
8,USD1USDT,0.999531,1.740512e+08,1.739695e+08,USD1
9,BFUSDUSDT,0.999376,8.415836e+06,8.410584e+06,BFUSD


In [16]:
universe = pd.merge(universe_wo_vol, stable_ex_df, on="quoteAsset", how="left")
universe["weightedAvgPrice"] = universe["weightedAvgPrice"].astype(float)
universe["volume"] = universe["volume"].astype(float)

universe["volume$"] = universe["weightedAvgPrice"] * universe["volume"] 


universe = universe[universe["volume$"]>100000]
universe.index.name = "no."
sorted_universe = universe.sort_values(by="weightedAvgPrice", ascending=False).reset_index(drop=True)
sorted_universe

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume,$vol_x,ex_symbol,usdtPrice,ex_volume,$vol_y,volume$
0,BTCFDUSD,BTC,FDUSD,61913.053470,9.304773e+02,5.760869e+07,FDUSDUSDT,0.998217,20087716.0,2.005191e+07,5.760869e+07
1,BTCUSDS,BTC,USDS,61798.826369,1.720270e+00,1.063107e+05,USDSUSDT,1.000566,317337.0,3.175166e+05,1.063107e+05
2,BTCUSD1,BTC,USD1,61763.497505,2.319084e+03,1.432348e+08,USD1USDT,0.999531,174051171.0,1.739695e+08,1.432348e+08
3,BTCEURI,BTC,EURI,53553.510367,2.556300e+00,1.368988e+05,EURIUSDT,1.155142,1730533.1,1.999012e+06,1.368988e+05
4,BTCEUR,BTC,EUR,53419.936211,3.521822e+02,1.881355e+07,EURUSDT,1.155644,19521458.9,2.255985e+07,1.881355e+07
5,ETHFDUSD,ETH,FDUSD,1640.552066,1.291600e+04,2.118937e+07,FDUSDUSDT,0.998217,20087716.0,2.005191e+07,2.118937e+07
6,ETHUSD1,ETH,USD1,1638.314807,4.558979e+04,7.469043e+07,USD1USDT,0.999531,174051171.0,1.739695e+08,7.469043e+07
7,ETHUSDS,ETH,USDS,1637.758503,6.282510e+01,1.028923e+05,USDSUSDT,1.000566,317337.0,3.175166e+05,1.028923e+05
8,ETHEURI,ETH,EURI,1421.458802,8.652420e+01,1.229906e+05,EURIUSDT,1.155142,1730533.1,1.999012e+06,1.229906e+05
9,ETHEUR,ETH,EUR,1416.709961,6.070593e+03,8.600270e+06,EURUSDT,1.155644,19521458.9,2.255985e+07,8.600270e+06


In [17]:
altcoins = ["BTC", "SOL", "ETH", "BNB", "ZEC", "XRP","SUI", "LINK"]
altcoin_symbols = [i+"USDT" for i in altcoins]
altcoin_symbols_df = all_vol[all_vol["symbol"].isin(altcoin_symbols)]
altcoin_symbols_df.rename(columns={"symbol":"usdt_ob", "weightedAvgPrice": "usdtAssetPrice", "volume": "ob2_volume"}, inplace=True)

altcoin_symbols_df["baseAsset"] = altcoin_symbols_df["usdt_ob"].str.replace("USDT", "", regex=False)

altcoin_symbols_df

,usdt_ob,usdtAssetPrice,ob2_volume,$vol,baseAsset
11,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08,BTC
12,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08,ETH
98,BNBUSDT,589.533878,1.117128e+05,6.585847e+07,BNB
306,XRPUSDT,1.118376,1.109527e+08,1.240869e+08,XRP
431,LINKUSDT,7.743119,1.896963e+06,1.468841e+07,LINK
477,ZECUSDT,430.482430,4.291913e+05,1.847593e+08,ZEC
779,SOLUSDT,64.389632,2.778440e+06,1.789027e+08,SOL
2205,SUIUSDT,0.746339,3.231317e+07,2.411657e+07,SUI


In [18]:
universe_w_tri_asset = pd.merge(sorted_universe, altcoin_symbols_df, on="baseAsset", how="left")
universe_w_tri_asset

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume,$vol_x,ex_symbol,usdtPrice,ex_volume,$vol_y,volume$,usdt_ob,usdtAssetPrice,ob2_volume,$vol
0,BTCFDUSD,BTC,FDUSD,61913.053470,9.304773e+02,5.760869e+07,FDUSDUSDT,0.998217,20087716.0,2.005191e+07,5.760869e+07,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08
1,BTCUSDS,BTC,USDS,61798.826369,1.720270e+00,1.063107e+05,USDSUSDT,1.000566,317337.0,3.175166e+05,1.063107e+05,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08
2,BTCUSD1,BTC,USD1,61763.497505,2.319084e+03,1.432348e+08,USD1USDT,0.999531,174051171.0,1.739695e+08,1.432348e+08,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08
3,BTCEURI,BTC,EURI,53553.510367,2.556300e+00,1.368988e+05,EURIUSDT,1.155142,1730533.1,1.999012e+06,1.368988e+05,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08
4,BTCEUR,BTC,EUR,53419.936211,3.521822e+02,1.881355e+07,EURUSDT,1.155644,19521458.9,2.255985e+07,1.881355e+07,BTCUSDT,61715.558143,1.565000e+04,9.658485e+08
5,ETHFDUSD,ETH,FDUSD,1640.552066,1.291600e+04,2.118937e+07,FDUSDUSDT,0.998217,20087716.0,2.005191e+07,2.118937e+07,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08
6,ETHUSD1,ETH,USD1,1638.314807,4.558979e+04,7.469043e+07,USD1USDT,0.999531,174051171.0,1.739695e+08,7.469043e+07,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08
7,ETHUSDS,ETH,USDS,1637.758503,6.282510e+01,1.028923e+05,USDSUSDT,1.000566,317337.0,3.175166e+05,1.028923e+05,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08
8,ETHEURI,ETH,EURI,1421.458802,8.652420e+01,1.229906e+05,EURIUSDT,1.155142,1730533.1,1.999012e+06,1.229906e+05,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08
9,ETHEUR,ETH,EUR,1416.709961,6.070593e+03,8.600270e+06,EURUSDT,1.155644,19521458.9,2.255985e+07,8.600270e+06,ETHUSDT,1636.529145,2.734128e+05,4.474481e+08


In [22]:
signal = ['symbol', 'baseAsset', 'weightedAvgPrice', 'usdtPrice' , 'volume$', 'usdt_ob','usdtAssetPrice', '$vol']
arb_in_universe = universe_w_tri_asset[signal].dropna()

arb_in_universe = arb_in_universe.astype({"weightedAvgPrice": float, "usdtPrice": float})

arb_in_universe["implied_usdt_price"] = arb_in_universe["weightedAvgPrice"] * arb_in_universe["usdtPrice"]

arb_in_universe

,symbol,baseAsset,weightedAvgPrice,usdtPrice,volume$,usdt_ob,usdtAssetPrice,$vol,implied_usdt_price
0,BTCFDUSD,BTC,61913.053470,0.998217,5.760869e+07,BTCUSDT,61715.558143,9.658485e+08,61802.686642
1,BTCUSDS,BTC,61798.826369,1.000566,1.063107e+05,BTCUSDT,61715.558143,9.658485e+08,61833.803887
2,BTCUSD1,BTC,61763.497505,0.999531,1.432348e+08,BTCUSDT,61715.558143,9.658485e+08,61734.526719
3,BTCEURI,BTC,53553.510367,1.155142,1.368988e+05,BTCUSDT,61715.558143,9.658485e+08,61861.920318
4,BTCEUR,BTC,53419.936211,1.155644,1.881355e+07,BTCUSDT,61715.558143,9.658485e+08,61734.406327
5,ETHFDUSD,ETH,1640.552066,0.998217,2.118937e+07,ETHUSDT,1636.529145,4.474481e+08,1637.627602
6,ETHUSD1,ETH,1638.314807,0.999531,7.469043e+07,ETHUSDT,1636.529145,4.474481e+08,1637.546339
7,ETHUSDS,ETH,1637.758503,1.000566,1.028923e+05,ETHUSDT,1636.529145,4.474481e+08,1638.685458
8,ETHEURI,ETH,1421.458802,1.155142,1.229906e+05,ETHUSDT,1636.529145,4.474481e+08,1641.987062
9,ETHEUR,ETH,1416.709961,1.155644,8.600270e+06,ETHUSDT,1636.529145,4.474481e+08,1637.211771


In [23]:
print(altcoin_symbols_df['usdt_ob'].tolist())

['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'LINKUSDT', 'ZECUSDT', 'SOLUSDT', 'SUIUSDT']


In [25]:
#arb_in_universe['alpha'] = abs((arb_in_universe['usdtAssetPrice'] - arb_in_universe['implied_usdt_price'])/arb_in_universe['usdtAssetPrice'])
# price difference doesn't matter how fast it differs matter

arb_in_universe['alpha_rate'] = abs((arb_in_universe['$vol'] - arb_in_universe['volume$']))

arb_in_universe.sort_values(by='alpha_rate', ascending=False).reset_index(drop=True)

,symbol,baseAsset,weightedAvgPrice,usdtPrice,volume$,usdt_ob,usdtAssetPrice,$vol,implied_usdt_price,alpha_rate
0,BTCUSDS,BTC,61798.826369,1.000566,1.063107e+05,BTCUSDT,61715.558143,9.658485e+08,61833.803887,9.657422e+08
1,BTCEURI,BTC,53553.510367,1.155142,1.368988e+05,BTCUSDT,61715.558143,9.658485e+08,61861.920318,9.657116e+08
2,BTCEUR,BTC,53419.936211,1.155644,1.881355e+07,BTCUSDT,61715.558143,9.658485e+08,61734.406327,9.470350e+08
3,BTCFDUSD,BTC,61913.053470,0.998217,5.760869e+07,BTCUSDT,61715.558143,9.658485e+08,61802.686642,9.082399e+08
4,BTCUSD1,BTC,61763.497505,0.999531,1.432348e+08,BTCUSDT,61715.558143,9.658485e+08,61734.526719,8.226138e+08
5,ETHUSDS,ETH,1637.758503,1.000566,1.028923e+05,ETHUSDT,1636.529145,4.474481e+08,1638.685458,4.473452e+08
6,ETHEURI,ETH,1421.458802,1.155142,1.229906e+05,ETHUSDT,1636.529145,4.474481e+08,1641.987062,4.473251e+08
7,ETHEUR,ETH,1416.709961,1.155644,8.600270e+06,ETHUSDT,1636.529145,4.474481e+08,1637.211771,4.388478e+08
8,ETHFDUSD,ETH,1640.552066,0.998217,2.118937e+07,ETHUSDT,1636.529145,4.474481e+08,1637.627602,4.262587e+08
9,ETHUSD1,ETH,1638.314807,0.999531,7.469043e+07,ETHUSDT,1636.529145,4.474481e+08,1637.546339,3.727576e+08


In [166]:
print(arb_in_universe.sort_values(by='alpha', ascending=False).reset_index(drop=True)["symbol"].to_list())

['ZECUSD1', 'XRPUSD1', 'XRPEUR', 'SUIFDUSD', 'SOLUSD1', 'XRPFDUSD', 'XRPRLUSD', 'SUIEUR', 'SOLFDUSD', 'BNBFDUSD', 'BTCFDUSD', 'SUIUSD1', 'ETHUSD1', 'ETHFDUSD', 'SOLEUR', 'ETHEUR', 'BNBUSD1', 'BTCEUR', 'BTCUSD1', 'LINKEUR', 'BNBEUR', 'BTCEURI']


In [171]:
print(arb_in_universe["usdt_ob"].unique())

<ArrowStringArray>
[ 'BTCUSDT',  'ETHUSDT',  'BNBUSDT',  'ZECUSDT',  'SOLUSDT', 'LINKUSDT',
  'XRPUSDT',  'SUIUSDT']
Length: 8, dtype: str


In [26]:
arb_in_universe[arb_in_universe['symbol'].isin(['EURIUSDT','BTCEURI','EURIUSDT'])]

,symbol,baseAsset,weightedAvgPrice,usdtPrice,volume$,usdt_ob,usdtAssetPrice,$vol,implied_usdt_price,alpha_rate
3,BTCEURI,BTC,53553.510367,1.155142,136898.83855,BTCUSDT,61715.558143,9.658485e+08,61861.920318,9.657116e+08
